# 02 Â· Daily cockpit on the simulated ledger

**SIMULATED** - `src/simulate.py`, seed 42. Not Emerging Travel Group data.

Order: data quality -> daily table with comparisons -> alerts -> incidents -> monthly bridge.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")   # keep outputs clean
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # reuse the project's scripts
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
from common import OUT, SIM, RAW, PUBLIC
import analysis as an
raw = an.load()
issues, c = an.dq_checks(raw)
issues

,issue,date_from,date_to,scope,rows,effect,action
0,Duplicated bookings,2026-07-21,2026-07-21,Channel API,152,"Day sales overstated by +27% (EUR 53,393)",Removed duplicates on booking_id; ask data eng...
1,Missing net cost,2026-08-03,2026-08-09,Supplier SUP-WS-A,124,"If read as zero, total margin rate shows 12.5%...",Cost imputed with the supplier's markup of the...
2,Time-zone shift in a feed,2026-10-12,2026-10-18,Channel API,60,Timestamps +5 h: 60 bookings counted on the wr...,Shifted back 5 h for the window; feed owner to...
3,"Sold below net cost (business alert, not a dat...",2025-05-01,2026-10-31,All bookings,1498,29% of them on SUP-FL-A,Sent to pricing: selling price below net cost ...
4,Selling price <= 0,,,All bookings,0,None,Check passed
5,Check-in before booking date,,,All bookings,0,None,Check passed
6,Cancellation before booking date,,,All bookings,0,None,Check passed
7,Missing FX rate,,,All bookings,0,None,Check passed


## 1. Daily totals with D-1, same day last year, 7-day average and expected value

In [2]:
daily = an.add_comparisons(an.daily_table(c))
daily.loc[daily.booking_date.between('2026-09-10', '2026-09-16'), ['booking_date', 'bookings', 'bookings_d1', 'bookings_ly', 'bookings_expected', 'sales_eur', 'sales_eur_expected', 'margin_rate', 'margin_rate_expected']].set_index('booking_date').round(3)

,bookings,bookings_d1,bookings_ly,bookings_expected,sales_eur,sales_eur_expected,margin_rate,margin_rate_expected
booking_date,,,,,,,,
2026-09-10,362,406.0,312.0,404.50,120658.76,138391.792,0.079,0.074
2026-09-11,338,362.0,305.0,356.25,116115.17,122876.265,0.077,0.074
2026-09-12,242,338.0,261.0,298.00,78567.10,105979.830,0.080,0.073
2026-09-13,293,242.0,273.0,315.25,101492.57,109224.402,0.080,0.074
2026-09-14,405,293.0,377.0,425.50,137922.22,140455.378,0.078,0.073
2026-09-15,380,405.0,343.0,417.50,123036.89,140719.315,0.077,0.075
2026-09-16,395,380.0,364.0,416.25,139054.37,146647.060,0.078,0.075


## 2. Alerts
Expected = same weekday over the previous 4 weeks; noise = standard deviation of the daily deviation over 56 days; alert at |z| >= 3.5, segments expected at 20+ bookings a day. The answer key (`events_truth.csv`) is used only to score the alerts.

In [3]:
al = pd.read_csv(OUT / 'sim' / 'alerts.csv')
print(len(al), 'alerts')
al['matched_event'].fillna('-').value_counts()

121 alerts


matched_event
-                          112
E4 UAE holiday               5
E1 supplier net rate         2
E6 Greece cancellations      1
E2 API partner lost          1
Name: count, dtype: int64

In [4]:
al[al['matched_event'].notna()]

,date,dimension,member,metric,actual,expected,z,matched_event
63,2026-05-12,client_country,AE,bookings,76.000000,39.000000,4.000150,E4 UAE holiday
64,2026-05-13,client_country,AE,sales_eur,23823.500000,12405.035000,3.615791,E4 UAE holiday
65,2026-05-14,client_country,AE,bookings,77.000000,35.750000,4.089785,E4 UAE holiday
66,2026-05-14,client_country,AE,sales_eur,28547.450000,12325.115000,4.699492,E4 UAE holiday
67,2026-05-17,client_country,AE,sales_eur,21072.280000,8166.185000,4.406123,E4 UAE holiday
86,2026-06-15,destination_country,ES,margin_rate,0.066977,0.086698,-3.893741,E1 supplier net rate
87,2026-06-18,destination_country,ES,margin_rate,0.068558,0.089371,-3.550531,E1 supplier net rate
96,2026-08-13,destination_country,EL,cancellations,40.000000,14.750000,3.736777,E6 Greece cancellations
100,2026-09-07,partner_id,API-002,margin_rate,0.030472,0.064093,-3.975937,E2 API partner lost


## 3. Incident cards

In [5]:
pd.read_csv(OUT / 'sim' / 'incidents.csv')[['event', 'title', 'evidence', 'impact_eur', 'owner']]

,event,title,evidence,impact_eur,owner
0,E1,Supplier net-rate increase,Margin rate on Bedbank C x Spain 8.5% -> 2.2%;...,-32555.540318,Supply / contracting
1,E2,Large API partner moved to a competitor,API-002 bookings/day 10 -> 0; it explains 55% ...,-10155.692857,Key account manager + pricing
2,E3,"Currency effect (real TRY depreciation, ECB ra...",TRY -12% vs EUR (Oct 26 vs Oct 25). Sep-Oct Yo...,-122233.653371,Pricing + finance (treasury)
3,E4,Source-market holiday (UAE),Bookings/day +93% vs the previous 4 weeks; sam...,12267.246429,Commercial team (market manager UAE)
4,E6,Cancellation spike at one destination (Greece),Cancellations/day for Greece 14 -> 30 (8-14 Au...,-4977.675000,Customer operations + supply
5,E7,Price-match requests,"1,347 requests; 21% discounted, 51% refused wi...",-2668.490000,Pricing analyst


## 4. Margin bridge, Sep 2025 -> Sep 2026: volume / mix / price / FX / cost

In [6]:
bridge, seg = an.margin_bridge(c, '2025-09', '2026-09', ['brand', 'line_of_business', 'client_country'])
bridge.round(0)

,step,eur
0,Margin 2025-09,231819.0
1,Volume,31055.0
2,Mix,-887.0
3,Price,55427.0
4,FX,-17315.0
5,Cost,-20838.0
6,Margin 2026-09,279261.0


In [7]:
seg.groupby('client_country')[['volume', 'mix', 'price', 'fx', 'cost']].sum().round(0).sort_values('fx')

,volume,mix,price,fx,cost
client_country,,,,,
TR,3615.0,-590.0,39394.0,-41253.0,5982.0
IN,2790.0,251.0,27460.0,-15942.0,-2498.0
PL,3033.0,-180.0,3683.0,-3776.0,-2013.0
ES,2426.0,64.0,374.0,0.0,-1726.0
IT,1723.0,-279.0,4451.0,0.0,-4781.0
NL,1528.0,-128.0,2705.0,0.0,-3211.0
FR,3421.0,719.0,-2948.0,0.0,2784.0
DE,6114.0,177.0,3583.0,0.0,-3954.0
GB,-3639.0,73.0,855.0,3724.0,-3666.0


The FX column is concentrated on the markets whose currency fell against the euro (real ECB rates), with Turkey first.